# Porto housing pressure: reproducible walkthrough

Observed municipality-year comparisons are separate from the **hypothetical** capacity and commuting illustrations. This notebook makes **no causal displacement estimate**.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from statistics import median
from analysis import load_panel, observed_summary, displacement_chain, commute_counterfactual, run
panel = load_panel()
summary = observed_summary(panel)
assert len(panel) == 119 and len(summary) == 17
print('Median of 17 municipal sale-price growth rates:', round(median(r['sale_price_growth_2019_2025_pct'] for r in summary), 1), '%')
print('Median of 17 municipal new-rent growth rates:', round(median(r['new_lease_rent_growth_2020_2025_pct'] for r in summary), 1), '%')

## Hypothetical households and capacity

The fractions below are **assumptions**, not observed Porto flows. Household totals must be conserved.

In [ ]:
for label, fractions in [('constrained', [0.2, 0.2, 0.25]), ('responsive', [0.55, 0.6, 0.75])]:
    stages = displacement_chain(1000, fractions)
    assert abs(sum(s.absorbed for s in stages) + stages[-1].onward - 1000) < 1e-9
    print(label, 'hypothetical households beyond tier 3:', stages[-1].onward)

## Generalized commuting cost

The illustrative monetary coefficients are **not Porto transport fares or measured time values**.

In [ ]:
for distance in (0, 10, 25, 40, 60):
    entry = commute_counterfactual(distance)
    print(distance, 'km:', round(entry['illustrative_total_eur_month'], 2), 'EUR/month')
with TemporaryDirectory() as scratch:
    metadata = run(Path(scratch))
    assert metadata['figure_count'] == 8 and not metadata['causal_effect_estimated']
print('Eight figures and three CSV outputs reproduced successfully.')